In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
##1. Load Dataset:

In [ ]:
df = pd.read_csv("powerplant_data.csv")

In [ ]:
df.head()
# AT => Temparature
# V => vaccum
# AP => pressure
# RH => Humidity
# PE => Produce energy

In [ ]:
df.isnull().sum()

In [ ]:
X = df.drop("PE", axis =1) # Features
y = df["PE"] # Labels

In [ ]:
X.head()

In [ ]:
## Train - Test- Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
df.shape

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [ ]:
## 2. Convert Dataset into Tensors

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled , dtype= torch.float32)
y_train_tensor = torch.tensor(y_train.values , dtype= torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype= torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype= torch.float32).view(-1, 1)

In [ ]:
## 3. Implement Tensor DataSet and DataLoader

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
### Create DataLoader from Dataset

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
## Deep Learning Part : 

In [ ]:
## 4. Define ANN Model

In [ ]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # 1st Hidden Layeer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),
    
            #2nd Hidden
            nn.Linear(6, 6),
            nn.ReLU(),
    
            # output layer
            nn.Linear(6,1),
        )
    # Forward propagation
    def forward(self, x):
        return self.model(x)
    

In [ ]:
model = ANN()

# loss, optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
##5 . Training ANN Model

In [ ]:
train_losses = []
val_losses = []

best_val_loss = float("inf") # infinity

epochs = 100
for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # total training loss of one epoch

    for xb, yb in train_loader: # This trainloader gives us every bacth
        # xb = features of 1 batch
        # yb = labels of 1 batch

        optimizer.zero_grad() #Clears all previous gradients before each training step.
        outputs = model(xb) # (forward propagation)... predicted outputs for this batch
        loss = criterion(outputs, yb) # it returns MSELoss
        loss.backward() # back prop... compute gradients
        optimizer.step() # params update using step function

        running_loss += loss.item() # loss is tensor value => py float
        
    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    #validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad(): # No gradients compute
        for xb , yb in test_loader:
            outputs = model(xb)
            loss = criterion(outputs, yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")
    

In [ ]:
loss_df = pd.DataFrame({
    "Training Loss": train_losses,
    "Validation Loss": val_losses
})

plt.plot(loss_df["Training Loss"], label = "Training Loss")
plt.plot(loss_df["Validation Loss"], label = "Validation Loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")
plt.legend()

In [ ]:
if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt") # save model parameters

In [ ]:
#loading the best model
model.load_state_dict(torch.load("best_model.pt"))

In [ ]:
## Model Evaluation